# 01. Extração, Transformação e Carga (ETL) e Modelagem Dimensional

### Documentação da Etapa
* **Objetivo da etapa:** Consolidar os microdados particionados e estruturar o modelo dimensional em 3 tabelas em memória (`df_transacional`, `df_dimensao_sh4` e `df_sazonal`).
* **Dados de entrada:** Arquivos CSV particionados por biênios em `data/raw/` (`2019-01_2020-12.csv`, `2021-01_2022-12.csv`, `2023-01_2024-12.csv`, `2025-01_2026-07.csv`).
* **Procedimentos realizados:**
  1. Leitura unificada e validação do volume de registros.
  2. Limpeza textual e conversão de valores monetários/peso (padrão pt-BR para `numeric`).
  3. Separação de atributos de calendário (`mes_numero`, `mes_nome`).
  4. Derivação da Tabela Dimensão Produto (`df_dimensao_sh4`), da Fato Transacional (`df_transacional`) e do Agregado Temporal (`df_sazonal`).
* **Resultados obtidos:** Criação das três estruturas tabulares em memória e exportação de `dados_completos_2019_2026.csv` em `data/processed/`.
* **Interpretação:** A base está estruturada de forma dimensional, otimizando o consumo de memória e preparando as agregações necessárias para as análises descritivas e econométricas.
* **Decisões metodológicas:** Manter as descrições textuais dos produtos isoladas na tabela dimensão para evitar redundância e ganho de eficiência no processamento.
* **Próximo passo:** Entendimento e validação da qualidade dos dados e recorte geográfico regional (`02_entendimento_dados.ipynb`).

In [12]:
# Configuração do diretório de trabalho e importação do pipeline ETL
if (basename(getwd()) == "notebooks") {
  setwd("..")
}

source("R/etl.R")

In [13]:
# 1. Definição dos caminhos das partições de entrada
arquivos_particionados <- c(
  "data/raw/2019-01_2020-12.csv",
  "data/raw/2021-01_2022-12.csv",
  "data/raw/2023-01_2024-12.csv",
  "data/raw/2025-01_2026-07.csv"
)
arquivo_processado <- "data/processed/dados_completos_2019_2026.csv"

# 2. Execução do pipeline que retorna a lista com os 3 DataFrames
modelo_dimensional <- executar_pipeline_etl(
  caminhos_particoes = arquivos_particionados,
  caminho_saida = arquivo_processado
)

# 3. Desempacotamento das 3 estruturas para o ambiente global
df_transacional <- modelo_dimensional$df_transacional
df_dimensao_sh4  <- modelo_dimensional$df_dimensao_sh4
df_sazonal       <- modelo_dimensional$df_sazonal

--- Lendo e validando arquivos de entrada ---

Arquivo '2019-01_2020-12.csv': 101936 linhas.



Arquivo '2021-01_2022-12.csv': 112721 linhas.

Arquivo '2023-01_2024-12.csv': 120210 linhas.

Arquivo '2025-01_2026-07.csv': 93187 linhas.


--- Aplicando limpeza e conversões de tipos ---

Arquivo consolidado salvo em: data/processed/dados_completos_2019_2026.csv


--- Construindo Estruturas Dimensionais ---

1. df_transacional : 428054 linhas.

2. df_dimensao_sh4  : 1051 produtos únicos.

3. df_sazonal       : 546 agregações temporais.



In [14]:
# Inspeção da Tabela Fato Transacional
glimpse(df_transacional)

Rows: 428,054
Columns: 9
$ Fluxo                <chr> "Exportação", "Exportação", "Exportação", "Export…
$ Ano                  <int> 2020, 2020, 2020, 2020, 2020, 2020, 2020, 2020, 2…
$ mes_numero           <int> 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1…
$ mes_nome             <chr> "Dezembro", "Dezembro", "Dezembro", "Dezembro", "…
$ Município            <chr> "Cubatão - SP", "Cubatão - SP", "Cubatão - SP", "…
$ País                 <chr> "Afeganistão", "Angola", "Arábia Saudita", "Barei…
$ `Código SH4`         <chr> "0207", "0207", "0207", "0207", "0207", "0207", "…
$ `Valor US$ FOB`      <dbl> 47408, 174573, 4598811, 223087, 25773, 11294, 785…
$ `Quilograma Líquido` <dbl> 27478, 155010, 2484207, 170415, 27000, 25725, 561…


In [15]:
# Inspeção da Tabela Dimensão Produto (SH4)
glimpse(df_dimensao_sh4)

Rows: 1,051
Columns: 2
$ `Código SH4`    <chr> "0101", "0102", "0103", "0104", "0105", "0106", "0201"…
$ `Descrição SH4` <chr> "Cavalos, asininos e muares, vivos", "Animais vivos da…


In [16]:
# Inspeção da Tabela Fato Agregada Sazonal
glimpse(df_sazonal)

Rows: 546
Columns: 7
$ Fluxo                <chr> "Exportação", "Exportação", "Exportação", "Export…
$ Ano                  <int> 2019, 2019, 2019, 2019, 2019, 2019, 2019, 2019, 2…
$ mes_numero           <int> 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6…
$ mes_nome             <chr> "Janeiro", "Janeiro", "Janeiro", "Fevereiro", "Fe…
$ Município            <chr> "Cubatão - SP", "Guarujá - SP", "Santos - SP", "C…
$ `Valor US$ FOB`      <dbl> 95422672, 33948175, 159338047, 69486182, 72886106…
$ `Quilograma Líquido` <dbl> 141577622, 72870251, 440201899, 117473896, 177422…


In [17]:
write_csv2(df_transacional, "data/processed/df_transacional.csv")
write_csv2(df_dimensao_sh4,  "data/processed/df_dimensao_sh4.csv")
write_csv2(df_sazonal,       "data/processed/df_sazonal.csv")